# Sentiment Analysis - YouTube Comments (Indonesian)
**Nama**: Asep Haryana Saputra

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout, SpatialDropout1D, BatchNormalization
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import resample
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm

print(f"TensorFlow Version: {tf.__version__}")

2026-02-10 19:50:53.418572: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 19:50:53.492887: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-02-10 19:50:55.046943: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow Version: 2.20.0


## 2. Data Loading & Preprocessing
**Target Classes**:
- 0: **Negative**
- 1: **Positive**
- 2: **Neutral**

In [2]:
# Nama file dataset
filename = 'dataset_youtube.csv'

if not os.path.exists(filename):
    raise FileNotFoundError(f"File {filename} not found! Please run the scraping script first.")

df = pd.read_csv(filename)
print(f"Loaded {len(df)} rows.")

if 'label' in df.columns and df['label'].isin([0,1,2]).all():
    print("Using existing labels from dataset.")
    df_clean = df.copy()
else:
    print("Labeling data using heuristic...")
    # Auto-Labeling Function (Heuristic)
    def auto_label(text):
        text = str(text).lower()
        
        # SPAM/Neutral Keywords (Adapted for mapping)
        spam_keywords = [
            'cek ig', 'promo', 'jual', 'obat', 'peninggi', 'pelangsing', 'subscribe', 'sub4sub', 
            'mampir', 'channel', 'link bio', 'giveaway', 'hadiah', 'gratis', 'follow', 'visit',
            'klik', 'top up', 'jasa', 'bisnis', 'wa', '08' 
        ]
        if any(k in text for k in spam_keywords):
            return 2 # Neutral/Spam
            
        # POSITIVE Keywords
        pos_keywords = [
            'bagus', 'keren', 'mantap', 'semangat', 'good', 'nice', 'love', 'thanks', 'terima kasih', 
            'bermanfaat', 'membantu', 'suka', 'cool', 'best', 'hebat', 'lanjutkan', 'panutan', 'idol',
            'legend', 'terhibur', 'lucu', 'ngakak', 'kocak', 'anjay', 'anjing' if 'keren' in text else '@@@', 
            'gokil', 'pecah', 'trending', 'karyamu', 'support', 'wow', 'gas', 'menyala'
        ]
        if any(k in text for k in pos_keywords):
            return 1 # Positive
            
        # NEGATIVE Keywords
        neg_keywords = [
            'jelek', 'buruk', 'sampah', 'gagal', 'benci', 'hate', 'bad', 'stupid', 'idiot', 'bodoh', 
            'gajelas', 'malas', 'susah', 'ribet', 'unsub', 'cringe', 'alay', 'lebay', 'norak', 
            'garing', 'toxic', 'kontol', 'memek', 'anjing', 'bangsat', 'tolol', 'goblok', 'hujat', 'drama',
            'settingan', 'sampah', 'kecewa', 'merugikan', 'penipu', 'bohong', 'palsu'
        ]
        if any(k in text for k in neg_keywords):
            return 0 # Negative
            
        return -1 # Unlabeled

    df['label'] = df['text'].apply(auto_label)
    df_clean = df[df['label'] != -1].copy()

print(f"Labeled Data Before Balancing: {len(df_clean)} rows")
print(df_clean['label'].value_counts())

# --- BALANCING DATA (Upsampling) ---
# Separate majority and minority classes
df_pos = df_clean[df_clean.label==1]
df_neg = df_clean[df_clean.label==0]
df_neu = df_clean[df_clean.label==2]

# Upsample minority classes
n_samples = max(len(df_pos), len(df_neg), len(df_neu)) # Match majority class

df_pos_upsampled = resample(df_pos, replace=True, n_samples=n_samples, random_state=42)
df_neg_upsampled = resample(df_neg, replace=True, n_samples=n_samples, random_state=42)
df_neu_upsampled = resample(df_neu, replace=True, n_samples=n_samples, random_state=42)

# Combine majority class with upsampled minority classes
df_balanced = pd.concat([df_pos_upsampled, df_neg_upsampled, df_neu_upsampled])
df_clean = df_balanced # Replace working dataframe

print("\nAfter Balancing:")
print(df_clean['label'].value_counts())

df_clean.head()

Loaded 11000 rows.
Using existing labels from dataset.
Labeled Data Before Balancing: 11000 rows
label
1    6416
0    3436
2    1148
Name: count, dtype: int64

After Balancing:
label
1    6416
0    6416
2    6416
Name: count, dtype: int64


,text,label
1451,terakhir saya makan pesan via delivery order ....,1
9243,"tempat nya enak , nyaman . variasi mie nya ban...",1
8937,"bagus lounge dengan pemandangan kota bandung ,...",1
8883,"1 st tim coba ke sini , kata nya tempat nya ba...",1
6396,"itu kenapa saya percaya , karena bisa menjaga ...",1


In [3]:
# Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df_clean['cleaned_text'] = df_clean['text'].apply(clean_text)
df_clean = df_clean[df_clean['cleaned_text'].str.len() > 3] # Min length filter

## 3. Modeling Schemes Comparison
We will test 3 different schemes:
1. **Traditional ML (SVM + TF-IDF)** - Target > 85%
2. **Alternative ML (Random Forest + CountVectorizer)** - Target > 85%
3. **Deep Learning (Bi-LSTM + Optimization)** - Target > 92%

### Scheme 1: SVM + TF-IDF (80/20 Tuning)

In [4]:
# Split Data
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
    df_clean['cleaned_text'], df_clean['label'], test_size=0.2, random_state=42, stratify=df_clean['label']
)

# Pipeline: TF-IDF -> SVM
pipeline_svm = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', C=1.0, probability=True))
pipeline_svm.fit(X_train_ml, y_train_ml)

# Verify Accuracy
y_pred_svm = pipeline_svm.predict(X_test_ml)
acc_svm = accuracy_score(y_test_ml, y_pred_svm)
print(f"Scheme 1 (SVM) Accuracy: {acc_svm:.4f}")
print(classification_report(y_test_ml, y_pred_svm, target_names=['Negative', 'Positive', 'Neutral']))

Scheme 1 (SVM) Accuracy: 0.9595
              precision    recall  f1-score   support

    Negative       0.94      0.95      0.95      1282
    Positive       0.96      0.93      0.95      1283
     Neutral       0.97      0.99      0.98      1283

    accuracy                           0.96      3848
   macro avg       0.96      0.96      0.96      3848
weighted avg       0.96      0.96      0.96      3848



### Scheme 2: Random Forest + CountVectorizer (80/20 Tuning)

In [5]:
# Pipeline: CountVectorizer -> Random Forest
pipeline_rf = make_pipeline(CountVectorizer(), RandomForestClassifier(n_estimators=100, random_state=42))
pipeline_rf.fit(X_train_ml, y_train_ml)

# Verify Accuracy
y_pred_rf = pipeline_rf.predict(X_test_ml)
acc_rf = accuracy_score(y_test_ml, y_pred_rf)
print(f"Scheme 2 (Random Forest) Accuracy: {acc_rf:.4f}")
print(classification_report(y_test_ml, y_pred_rf, target_names=['Negative', 'Positive', 'Neutral']))

Scheme 2 (Random Forest) Accuracy: 0.9712
              precision    recall  f1-score   support

    Negative       0.96      0.96      0.96      1282
    Positive       0.97      0.96      0.96      1283
     Neutral       0.98      1.00      0.99      1283

    accuracy                           0.97      3848
   macro avg       0.97      0.97      0.97      3848
weighted avg       0.97      0.97      0.97      3848



### Scheme 3: Deep Learning (Bi-LSTM)

In [6]:
VOCAB_SIZE = 5000
MAX_LENGTH = 100
EMBEDDING_DIM = 64

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df_clean['cleaned_text'])

sequences = tokenizer.texts_to_sequences(df_clean['cleaned_text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

X_dl = padded_sequences
y_dl = df_clean['label'].values

# Split Data (70/30 for Deep Learning Variation)
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_dl, y_dl, test_size=0.3, random_state=42, stratify=y_dl)
print(f"DL Train shape: {X_train_dl.shape}, Test shape: {X_test_dl.shape}")

DL Train shape: (13467, 100), Test shape: (5772, 100)


In [7]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax') # 3 Output Classes
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Callbacks
checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)

history = model.fit(
    X_train_dl, y_train_dl,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_dl, y_test_dl),
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/20


/home/asephs/venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1770727939.513019  786679 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3220 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


2026-02-10 19:52:22.839978: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002



  1/421 ━━━━━━━━━━━━━━━━━━━━ 26:22 4s/step - accuracy: 0.3125 - loss: 1.0977


  5/421 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.3266 - loss: 1.0969 


  9/421 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.3490 - loss: 1.0939

... (1,048 outputs left out)


417/421 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9958 - loss: 0.0130


Epoch 11: val_accuracy improved from 0.96500 to 0.96622, saving model to best_model.keras



Epoch 11: finished saving model to best_model.keras



421/421 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9963 - loss: 0.0113 - val_accuracy: 0.9662 - val_loss: 0.2446 - learning_rate: 2.5000e-04


In [8]:
# Load best model
best_model = load_model("best_model.keras")
loss, acc_dl = best_model.evaluate(X_test_dl, y_test_dl)
print(f"Scheme 3 (Bi-LSTM) Accuracy: {acc_dl:.4f}")

y_pred_probs = best_model.predict(X_test_dl)
y_pred_dl = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test_dl, y_pred_dl, target_names=['Negative', 'Positive', 'Neutral']))


  1/181 ━━━━━━━━━━━━━━━━━━━━ 1:28 490ms/step - accuracy: 1.0000 - loss: 0.0025


 10/181 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9811 - loss: 0.1319    


 18/181 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9789 - loss: 0.1351


 26/181 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9777 - loss: 0.1434


 35/181 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9756 - loss: 0.1577


 44/181 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9740 - loss: 0.1719

... (33 outputs left out)


175/181 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


181/181 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


              precision    recall  f1-score   support

    Negative       0.95      0.96      0.96      1922
    Positive       0.97      0.94      0.96      1925
     Neutral       0.98      0.99      0.99      1925

    accuracy                           0.97      5772
   macro avg       0.97      0.97      0.97      5772
weighted avg       0.97      0.97      0.97      5772



## 4. Comparison of Results

In [9]:
results = pd.DataFrame({
    'Model': ['SVM + TF-IDF', 'RF + BOW', 'Bi-LSTM'],
    'Accuracy': [acc_svm, acc_rf, acc_dl],
    'Target > 85%': [acc_svm > 0.85, acc_rf > 0.85, acc_dl > 0.85],
    'DeepTarget > 92%': [False, False, acc_dl > 0.92]
})
print(results)

          Model  Accuracy  Target > 85%  DeepTarget > 92%
0  SVM + TF-IDF  0.959459          True             False
1      RF + BOW  0.971154          True             False
2       Bi-LSTM  0.966216          True              True


## 5. Inference / Testing
Test the model with custom sentences.

In [ ]:
def predict_sentiment(text_list):
    # Preprocess
    cleaned_texts = [clean_text(t) for t in text_list]
    sequences = tokenizer.texts_to_sequences(cleaned_texts)
    padded = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
    
    # Predict
    probs = best_model.predict(padded)
    preds = np.argmax(probs, axis=1)
    
    labels = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
    results = []
    for t, p in zip(text_list, preds):
        results.append({'Text': t, 'Prediction': labels[p]})
    
    return pd.DataFrame(results)

# Test Cases
test_sentences = [
    "Video ini sangat bermanfaat, terima kasih bang!",
    "Haha,keren",
    "Konten sampah, tidak mendidik sama sekali.",
    "Cek ig kami kak, ada promo menarik.",
    "Biasa saja sih, tidak ada yang spesial.",
    "Gajelas lu bang, unsub ah."
]

inference_results = predict_sentiment(test_sentences)
print(inference_results)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


                                              Text Prediction
0  Video ini sangat bermanfaat, terima kasih bang!    Neutral
1       Konten sampah, tidak mendidik sama sekali.   Negative
2              Cek ig kami kak, ada promo menarik.    Neutral
3          Biasa saja sih, tidak ada yang spesial.   Negative
4                       Gajelas lu bang, unsub ah.   Negative
